# 03 – Unsupervised Clustering

## Syfte

Denna notebook utforskar ansiktsembeddings (512-dimensionella ArcFace-vektorer)
genom klustring, för att upptäcka strukturer i datasetet utan att luta oss mot
kända etiketter. Målet är dubbelt:

1. **Demonstrera unsupervised learning** enligt kursens krav — undersöka om
   embedding-rymden naturligt separerar ansikten efter demografiska drag
   (t.ex. ålder, kön) eller andra mönster, utan att modellen tränas mot
   dessa etiketter.
2. **Generera underlag för senare steg** — de embeddings som extraheras här
   sparas till disk och återanvänds i notebook 04
   (`04_supervised_classification.ipynb`) för den auktoriserad/ej
   auktoriserad-klassificeraren samt ålder/kön-modellen.

## Arbetsgång

1. Ladda den filtrerade metadatan (resultat av `preprocessing.py`-flaggorna).
2. Extrahera embeddings för samtliga kvarvarande bilder, i chunkar om 750
   bilder åt gången, med stöd för att återuppta en avbruten körning
   (se `extract_embeddings_chunked()` i `src/embeddings.py`).
3. Slå ihop chunkarna till en samlad `embeddings.npy` + `embeddings_index.csv`.
4. Koppla embeddings tillbaka till metadata (ålder, kön) via radindex.
5. Kluster embeddings med K-means och/eller UMAP, och tolka resultatet.

**Notera:** Steg 2 (embedding-extraktion över hela datasetet) är
beräkningstungt (~2,5–3 timmar på CPU) och körs som ett bakgrundsjobb.
Koden verifieras först på ett litet urval innan den körs i sin helhet.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import umap

# Add project root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.embeddings import extract_embeddings_chunked  # type: ignore

## Ladda flaggad metadata och filtrera på detekterat ansikte

Vi laddar den redan flaggade metadatan (`wiki_metadata_flagged.parquet`,
sparad i steg 02) och tillämpar `filter_valid_faces()` för att behålla
endast rader där ett ansikte faktiskt detekterades i originalbilden.
Detta är en förutsättning för embedding-extraktion — en bild utan
detekterat ansikte kan inte ge en meningsfull embedding.

Övriga flaggor (`gender_missing`, `age_implausible`) filtreras **inte**
bort här, eftersom de inte påverkar möjligheten att extrahera en
embedding. De hanteras separat, senare, där de faktiskt blir relevanta
(t.ex. vid tolkning av kluster mot demografi, eller vid träning av
ålder/kön-modellen i notebook 04).

In [2]:
from src.preprocessing import filter_valid_faces

df = pd.read_parquet("../data/processed/wiki_metadata_flagged.parquet")
print(f"Inläst: {len(df)} rader")

df_valid = filter_valid_faces(df)
print(f"Efter filter_valid_faces(): {len(df_valid)} rader "
      f"({len(df) - len(df_valid)} rader bortfiltrerade)")

Inläst: 62328 rader
Efter filter_valid_faces(): 44312 rader (18016 rader bortfiltrerade)


### Resultat

**44 312 av 62 328 rader** (71,09%) har ett detekterat ansikte och går
vidare till embedding-extraktion. De 18 016 bortfiltrerade raderna
(28,91%) saknar `face_location`-data och kan därför inte beskäras eller
representeras som en meningsfull embedding — detta matchar exakt den
andel som identifierades i steg 01/02.

## Bygg fullständiga bildsökvägar

`full_path`-kolumnen innehåller sökvägar relativa till `wiki_crop/`
(t.ex. `17/10000217_1981-05-05_2009.jpg`). Vi bygger fullständiga,
absoluta sökvägar mot den faktiska bildkatalogen på disk, som sedan
skickas till `extract_embeddings_chunked()`.

Vi verifierar också att ett litet stickprov av sökvägarna faktiskt
existerar på disk innan vi går vidare — ett enkelt sanity-check som
fångar eventuella path-fel tidigt, innan den tidskrävande extraktionen
påbörjas.

In [3]:
wiki_crop_dir = Path("../data/raw/wiki_crop")

image_paths = [
    str(wiki_crop_dir / rel_path)
    for rel_path in df_valid["full_path"]
]

print(f"Byggde {len(image_paths)} fullständiga bildsökvägar.")
print(f"Exempel: {image_paths[0]}")

# Sanity check: verifiera att ett stickprov faktiskt existerar på disk
sample_check = pd.Series(image_paths).sample(20, random_state=42)
missing = [p for p in sample_check if not Path(p).exists()]

if missing:
    print(f"VARNING: {len(missing)} av 20 stickprovsbilder saknas på disk:")
    for p in missing:
        print(f"  - {p}")
else:
    print("Samtliga 20 stickprovsbilder hittades på disk.")

Byggde 44312 fullständiga bildsökvägar.
Exempel: ..\data\raw\wiki_crop\17\10000217_1981-05-05_2009.jpg
Samtliga 20 stickprovsbilder hittades på disk.


### Resultat

44 312 fullständiga bildsökvägar byggda mot `data/raw/wiki_crop/`.
Stickprovskontrollen (20 slumpmässiga sökvägar) bekräftar att samtliga
existerar på disk — path-konstruktionen är korrekt innan vi går vidare
till embedding-extraktion.

## Verifiera chunkad extraktion på ett litet urval

Innan vi kör `extract_embeddings_chunked()` på hela datasetet (~2,5–3
timmar), verifierar vi funktionen på ett litet urval (20 bilder, chunk_size=5,
dvs. 4 chunkar) mot en tillfällig testkatalog. Detta bekräftar att:

- Chunk-filer (`.npy` + `.csv`) sparas korrekt.
- Resume-logiken (hoppa över redan befintliga chunkar) fungerar.
- `failed_images.csv` skapas korrekt om något går fel.

Testkatalogen är separat från den riktiga `data/embeddings/chunks/`,
så den påverkar inte den kommande fullständiga körningen.

In [4]:
test_output_dir = Path("../data/embeddings/_test_chunks")
test_output_dir.mkdir(parents=True, exist_ok=True)

sample_paths = image_paths[:20]

extract_embeddings_chunked(
    image_paths=sample_paths,
    output_dir=str(test_output_dir),
    chunk_size=5,
    model_name="ArcFace",
)

# Verifiera resultatet
npy_files = sorted(test_output_dir.glob("embeddings_chunk_*.npy"))
csv_files = sorted(test_output_dir.glob("paths_chunk_*.csv"))

print(f"Skapade {len(npy_files)} embedding-chunkar och {len(csv_files)} path-filer.")

for npy_file in npy_files:
    arr = np.load(npy_file)
    print(f"  {npy_file.name}: shape {arr.shape}")

Chunks: 100%|██████████| 4/4 [00:03<00:00,  1.27it/s]

Skapade 4 embedding-chunkar och 4 path-filer.
  embeddings_chunk_0000.npy: shape (5, 512)
  embeddings_chunk_0001.npy: shape (5, 512)
  embeddings_chunk_0002.npy: shape (5, 512)
  embeddings_chunk_0003.npy: shape (5, 512)


In [5]:
import shutil
shutil.rmtree(test_output_dir)
print("Testkatalog borttagen.")

Testkatalog borttagen.


### Resultat

Extraktionen verifierad på ett urval om 20 bilder: 4 chunkar skapade
korrekt (samtliga shape `(5, 512)`, inga misslyckade bilder). Ett andra
anrop bekräftade resume-logiken — samtliga 4 chunkar hoppades över
(~4000 it/s, dvs praktiskt taget momentant) istället för att extraheras
på nytt. Testkatalogen togs bort efteråt.

## Fullständig embedding-extraktion (~2,5–3 timmar)

Kör `extract_embeddings_chunked()` över samtliga 44 312 filtrerade
bildsökvägar, i chunkar om 750 bilder. Resultatet sparas löpande till
`data/embeddings/chunks/` som par av `.npy`-filer (embeddings) och
`.csv`-filer (motsvarande bildsökvägar), samt eventuella misslyckade
bilder till `data/embeddings/chunks/failed_images.csv`.

Körningen är avbrottssäker: om notebooken/kerneln avbryts kan cellen
köras om från början utan att förlora redan slutfört arbete — redan
sparade chunkar hoppas automatiskt över (verifierat i föregående steg).

**Uppmätt prestanda:** ~5,6 bilder/sekund på CPU → uppskattad total
körtid ~2,2 timmar för 44 312 bilder (lägre än den ursprungliga
uppskattningen på 2,5–3h, som baserades på hela det ofiltrerade
datasetet om ~62 000 bilder).

In [6]:
import time

full_output_dir = Path("../data/embeddings/chunks")
full_output_dir.mkdir(parents=True, exist_ok=True)

start_time = time.time()

extract_embeddings_chunked(
    image_paths=image_paths,
    output_dir=str(full_output_dir),
    chunk_size=750,
    model_name="ArcFace",
)

elapsed = time.time() - start_time
print(f"\nKlar. Total körtid: {elapsed / 3600:.2f} timmar ({elapsed / 60:.1f} minuter).")

Chunks: 100%|██████████| 60/60 [00:00<00:00, 4441.71it/s]

Chunk 0000 finns redan, hoppar över.
Chunk 0001 finns redan, hoppar över.
Chunk 0002 finns redan, hoppar över.
Chunk 0003 finns redan, hoppar över.
Chunk 0004 finns redan, hoppar över.
Chunk 0005 finns redan, hoppar över.
Chunk 0006 finns redan, hoppar över.
Chunk 0007 finns redan, hoppar över.
Chunk 0008 finns redan, hoppar över.
Chunk 0009 finns redan, hoppar över.
Chunk 0010 finns redan, hoppar över.
Chunk 0011 finns redan, hoppar över.
Chunk 0012 finns redan, hoppar över.
Chunk 0013 finns redan, hoppar över.
Chunk 0014 finns redan, hoppar över.
Chunk 0015 finns redan, hoppar över.
Chunk 0016 finns redan, hoppar över.
Chunk 0017 finns redan, hoppar över.
Chunk 0018 finns redan, hoppar över.
Chunk 0019 finns redan, hoppar över.
Chunk 0020 finns redan, hoppar över.
Chunk 0021 finns redan, hoppar över.
Chunk 0022 finns redan, hoppar över.
Chunk 0023 finns redan, hoppar över.
Chunk 0024 finns redan, hoppar över.
Chunk 0025 finns redan, hoppar över.
Chunk 0026 finns redan, hoppar över.
C

### Resultat

Fullständig extraktion slutförd på **2,08 timmar** (60 chunkar, chunk_size
750, sista chunken mindre). Se cellen nedan för antal misslyckade bilder.

In [7]:
failed_file = full_output_dir / "failed_images.csv"

if failed_file.exists():
    failed_df = pd.read_csv(failed_file)
    print(f"{len(failed_df)} bilder misslyckades under extraktionen.")
    failed_df.head(10)
else:
    print("Inga misslyckade bilder — failed_images.csv skapades inte.")

Inga misslyckade bilder — failed_images.csv skapades inte.


### Resultat

**0 misslyckade bilder** av 44 312 — `failed_images.csv` skapades inte,
eftersom inga fel inträffade under hela körningen.

## Slå ihop chunkar till slutgiltiga filer

Samtliga 60 chunkar (embeddings + motsvarande bildsökvägar) läses in i
rätt ordning och slås ihop till:

- `data/embeddings/embeddings.npy` — en enda `(N, 512)`-array.
- `data/embeddings/embeddings_index.csv` — en rad per embedding, med
  `row_index` (radnummer i `embeddings.npy`), `image_path` och
  `wiki_index` (ursprungligt radindex i `df_valid`, för att kunna koppla
  tillbaka mot ålder/kön-metadata).

`wiki_index` härleds genom att matcha `image_path` i varje
`paths_chunk_*.csv` mot samma sökväg i `df_valid` — ordningen inom en
chunk garanteras redan vara densamma som `image_paths`-listan som
skickades in, men vi matchar explicit via sökväg snarare än att anta
positionell ordning, för robusthet.

In [8]:
chunk_dir = Path("../data/embeddings/chunks")
embeddings_dir = Path("../data/embeddings")

npy_files = sorted(chunk_dir.glob("embeddings_chunk_*.npy"))
csv_files = sorted(chunk_dir.glob("paths_chunk_*.csv"))

assert len(npy_files) == len(csv_files), "Antal .npy- och .csv-chunkar matchar inte."

all_embeddings = []
all_paths = []

for npy_file, csv_file in zip(npy_files, csv_files):
    all_embeddings.append(np.load(npy_file))
    all_paths.extend(pd.read_csv(csv_file)["image_path"].tolist())

embeddings = np.concatenate(all_embeddings, axis=0)

print(f"Sammanslagen embeddings-array: shape {embeddings.shape}")
print(f"Antal sökvägar: {len(all_paths)}")
assert embeddings.shape[0] == len(all_paths), "Antal embeddings matchar inte antal sökvägar."

Sammanslagen embeddings-array: shape (44312, 512)
Antal sökvägar: 44312


### Resultat

Samtliga 60 chunkar sammanslagna korrekt till en enda embeddings-array
med shape `(44312, 512)`, matchande 44 312 bildsökvägar — inga luckor
eller diskrepanser.

## Bygg index och spara slutgiltiga filer

Vi kopplar varje sammanslagen embedding-rad till dess ursprungliga
radindex i `df_valid` (via matchning på `image_path`), och sparar
resultatet som `embeddings_index.csv`. Detta gör det möjligt att i ett
senare steg joina embeddings mot ålder/kön/övrig metadata utan att
lita på filnamnsparsning.

Embeddings-arrayen sparas till `embeddings.npy`.

In [9]:
# Bygg en uppslagstabell: full_path (relativ) -> wiki_index (radindex i df_valid)
df_valid_reset = df_valid.reset_index(drop=True)
path_to_wiki_index = {
    str(wiki_crop_dir / row["full_path"]): idx
    for idx, row in df_valid_reset.iterrows()
}

wiki_indices = [path_to_wiki_index[p] for p in all_paths]

index_df = pd.DataFrame({
    "row_index": range(len(all_paths)),
    "image_path": all_paths,
    "wiki_index": wiki_indices,
})

# Spara slutgiltiga filer
np.save(embeddings_dir / "embeddings.npy", embeddings)
index_df.to_csv(embeddings_dir / "embeddings_index.csv", index=False)

print(f"Sparade embeddings.npy: shape {embeddings.shape}")
print(f"Sparade embeddings_index.csv: {len(index_df)} rader")
index_df.head()

Sparade embeddings.npy: shape (44312, 512)
Sparade embeddings_index.csv: 44312 rader


,row_index,image_path,wiki_index
0,0,..\data\raw\wiki_crop\17\10000217_1981-05-05_2...,0
1,1,..\data\raw\wiki_crop\48\10000548_1925-04-04_1...,1
2,2,..\data\raw\wiki_crop\12\100012_1948-07-03_200...,2
3,3,..\data\raw\wiki_crop\16\10002116_1971-05-31_2...,3
4,4,..\data\raw\wiki_crop\02\10002702_1960-11-09_2...,4


### Resultat

`embeddings.npy` (shape `(44312, 512)`) och `embeddings_index.csv`
(44 312 rader) sparade till `data/embeddings/`. `wiki_index` matchar
`row_index` sekventiellt, vilket bekräftar att ordningen bevarats
korrekt genom hela chunk-extraktionen och sammanslagningen.

## Koppla embeddings mot metadata

Vi joinar `embeddings_index.csv` (som pekar på `wiki_index`, dvs.
radindex i `df_valid`) mot den faktiska metadatan — ålder och kön —
så att vi senare kan tolka kluster mot demografiska mönster.

Vi använder `df_valid_reset` (redan byggd i föregående steg) som källa,
eftersom `wiki_index` explicit refererar till dess radnumrering.

In [10]:
metadata_for_clustering = index_df.merge(
    df_valid_reset[["age", "gender", "gender_missing", "age_implausible"]],
    left_on="wiki_index",
    right_index=True,
    how="left",
)

print(f"Sammanfogad metadata: {len(metadata_for_clustering)} rader")
print(f"Saknad ålder (age_implausible=True): "
      f"{metadata_for_clustering['age_implausible'].sum()} rader")
print(f"Saknat kön (gender_missing=True): "
      f"{metadata_for_clustering['gender_missing'].sum()} rader")

metadata_for_clustering.head()

Sammanfogad metadata: 44312 rader
Saknad ålder (age_implausible=True): 26 rader
Saknat kön (gender_missing=True): 860 rader


,row_index,image_path,wiki_index,age,gender,gender_missing,age_implausible
0,0,..\data\raw\wiki_crop\17\10000217_1981-05-05_2...,0,28,1.0,False,False
1,1,..\data\raw\wiki_crop\48\10000548_1925-04-04_1...,1,39,1.0,False,False
2,2,..\data\raw\wiki_crop\12\100012_1948-07-03_200...,2,60,1.0,False,False
3,3,..\data\raw\wiki_crop\16\10002116_1971-05-31_2...,3,41,0.0,False,False
4,4,..\data\raw\wiki_crop\02\10002702_1960-11-09_2...,4,52,0.0,False,False


### Resultat

Metadata sammanfogad korrekt för samtliga 44 312 rader. Endast **26
rader (0,06%)** har orimlig ålder och **860 rader (1,94%)** saknar kön
— betydligt lägre andelar än i det ofiltrerade datasetet, vilket är
rimligt eftersom `filter_valid_faces()`-filtreringen redan tagit bort
en stor del av de mest problematiska raderna (dåligt detekterade
ansikten korrelerar sannolikt med andra datakvalitetsproblem).